# 1.2 Prompt Templates

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Your API key - pick ONE option

Some cells in this notebook call a model, and you need your own API key. Run only ONE of the next two cells, whichever fits you:

**Option A - Gemini (free, recommended if you have no OpenAI credit):** Google AI Studio gives a free API key, no card needed. Open https://aistudio.google.com/apikey , create a key, then run the Gemini cell below. The Flash models run on a free tier with rate limits.

**Option B - OpenAI (paid):** create a key at https://platform.openai.com/api-keys and run the OpenAI cell below.

Whichever key you paste is stored only in this notebook session, never saved to the file.

In [ ]:
# Option A: Gemini (free). Run this cell OR the OpenAI one, not both.
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini (AI Studio) API key (input stays hidden): ")

In [ ]:
# Option B: OpenAI (paid). Run this cell OR the Gemini one, not both.
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Using Gemini instead of OpenAI in the cells below

If you ran the Gemini key cell, one more small step: wherever a cell below creates the model with `ChatOpenAI(...)`, swap that creation line for:

```python
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
```

(Use whatever variable name the cell used, for example `model = ...` instead of `llm = ...`.) Everything else stays the same: prompts, chains, parsers and tools do not care which provider the model came from. For embeddings, swap `OpenAIEmbeddings` for `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")` from the same package. On the OpenAI path, ignore this note and run the cells exactly as written.

## Learning objectives

- Explain what `ChatPromptTemplate` produces and how `from_template` differs from `from_messages`.
- Use the three message roles (system, human, ai) correctly and explain what each one is for.
- Reuse one template across different models and providers without changing it.
- Fill in stable values early with `.partial()`, including values computed at call time.
- Build few-shot prompts with `FewShotChatMessagePromptTemplate` and avoid the common formatting traps.

## The idea in plain words

A prompt template is a recipe for building the message list you send to a model. You write the shape once, with `{named_holes}` for the parts that change, and each call fills the holes with real values. The template itself is provider-agnostic: it produces plain message objects, and any chat model can consume them.

Why templates instead of f-strings? Three reasons. First, roles: a chat model does not take one string, it takes a list of messages with roles, and templates handle that structure. Second, validation: the template knows its variables, so a missing value fails fast with a clear error instead of silently sending a broken prompt. Third, composition: templates snap into chains, support partial filling, and nest inside each other (few-shot blocks slot into a bigger template). F-strings give you none of that.

## from_template: the quick way

`from_template` builds the most common case: a single human message with variables.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("Translate to French: {text}")
print(prompt.input_variables)  # ['text']

llm = ChatOpenAI(model="gpt-4o-mini")
chain = prompt | llm | StrOutputParser()
print(chain.invoke({"text": "Good morning"}))
# Bonjour

Under the hood this is one `HumanMessagePromptTemplate`. Rendering with `.format_messages(text="...")` produces a list with a single human message, which is what the model receives. Use `from_template` whenever the whole prompt is one user turn with a few slots. Reach for something richer the moment you need roles beyond "human".

## from_messages: full control

`from_messages` takes a list where each entry becomes one message. Entries can be tuples of `(role, template)`, message objects, other prompt templates, or placeholders:

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Answer in at most two sentences."),
    ("human", "{question}"),
    ("ai", "Understood. I am a {role} and will keep answers short."),
    ("human", "One more thing: {follow_up}"),
])
print(prompt.input_variables)  # ['follow_up', 'question', 'role']

for m in prompt.format_messages(role="librarian", question="Where is the poetry section?", follow_up="Do you lend e-books?"):
    print(f"[{m.type}] {m.content}")
# [system] You are a librarian. Answer in at most two sentences.
# [human] Where is the poetry section?
# [ai] Understood. I am a librarian and will keep answers short.
# [human] One more thing: Do you lend e-books?

The three roles each have a job:

- **system** sets the behaviour contract: who the model is, what rules it follows, what it must never do. This is the most important line in any prompt and the first thing to rewrite when output goes wrong.
- **human** carries the actual task input. There can be several; conversation history is human/ai pairs.
- **ai** is for scripted model turns: confirmations, worked examples, or the opening line of a few-shot pattern. The model reads these as "this is how I have been responding", which steers its style strongly.

Two more entry types you will use constantly:

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import SystemMessage, HumanMessage

prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="You are a concise support assistant."),  # static message object, no variables
    MessagesPlaceholder("history"),   # a whole message list injected at call time
    ("human", "{question}"),          # tuple form: variables get formatted
])

`MessagesPlaceholder("history")` is how chat history enters a template in Modules 2 and 4: the caller passes a list of message objects under the key `history`, and the placeholder splices them in as real messages. Message objects (as opposed to tuples) are for static text with no variables in it.

## Reusing one template across models

Because a template renders to plain message lists, the template never cares which provider answers. This chain runs unchanged if you swap the model:

In [ ]:
# Same prompt, OpenAI
chain = prompt | ChatOpenAI(model="gpt-4o-mini") | StrOutputParser()

# Same prompt, a local model via Ollama (needs: pip install langchain-ollama)
from langchain_ollama import ChatOllama
chain = prompt | ChatOllama(model="llama3.1") | StrOutputParser()

The provider-agnostic form is `init_chat_model("openai:gpt-4o-mini")` or `init_chat_model("ollama:llama3.1")`, which picks the right integration class. One caveat when you go cross-provider: not every provider treats the system role identically (some fold it into the first human message), so keep system instructions self-contained and avoid tricks that depend on one provider's quirks. Lesson 0.4 covered prompt wording for agents; the structural point here is that the template is the stable layer and the model is the swappable one.

## partial(): fill in what rarely changes

`.partial()` returns a new template with some variables already filled. Use it for values that are constant across many calls, like the assistant's name, the date, or the tone:

In [ ]:
base = ChatPromptTemplate.from_messages([
    ("system", "You are {assistant_name}, a {role}. Today is {today}."),
    ("human", "{question}"),
])

daily = base.partial(assistant_name="Ada", role="support agent")
print(daily.input_variables)  # ['question', 'today']

Two details worth knowing. First, `.partial()` returns a new template; the original is unchanged. Second, partial values can be callables, which are evaluated fresh on every format call, perfect for "today's date" or a value fetched from config:

In [ ]:
from datetime import date

daily = base.partial(
    assistant_name="Ada",
    role="support agent",
    today=lambda: date.today().isoformat(),  # evaluated at call time, not now
)
print(daily.format_messages(question="Hi")[0].content)
# You are Ada, a support agent. Today is 2026-10-20.

Watch the spelling of partial keys: a key that does not exist in the template is accepted silently and simply never fills anything, so a typo in a partial key fails quietly. Read the remaining `input_variables` after partialling; they should be exactly the keys you still plan to pass at call time.

## Few-shot prompts

Few-shot prompting shows the model input/output pairs so it copies the pattern. `FewShotChatMessagePromptTemplate` formats a list of examples through a small sub-template, then slots the result into a bigger template:

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "The invoice total is wrong", "output": "billing"},
    {"input": "The app crashes on login", "output": "bug"},
    {"input": "Add dark mode please", "output": "feature"},
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}"),
])

few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final = ChatPromptTemplate.from_messages([
    ("system", "Classify the ticket into exactly one category: billing, bug, or feature. Reply with the category only."),
    few_shot,                      # the example block lands here
    ("human", "{ticket}"),
])

for m in final.format_messages(ticket="I was charged twice"):
    print(f"[{m.type}] {m.content}")
# [system] Classify the ticket ...
# [human] The invoice total is wrong
# [ai] billing
# [human] The app crashes on login
# [ai] bug
# [human] Add dark mode please
# [ai] feature
# [human] I was charged twice

Notice the example block renders as alternating human/ai messages, exactly the shape a conversation would have. Each example is a dict whose keys fill the `example_prompt` variables. If your examples should be chosen dynamically (the nearest ones to the current input, say), pass `example_selector` instead of `examples`; the static list is the common case and what this course uses.

A practical rule: 3 to 5 examples is the sweet spot. One example under-specifies the pattern, ten examples burn tokens and the model starts copying quirks of the examples rather than the pattern. And make the examples diverse: three examples that all look alike teach less than three that each stress a different edge.

## Formatting pitfalls

These four traps account for most prompt-template debugging time. All were verified against the pinned versions:

1. **Unescaped braces become variables.** Any `{...}` in your text is treated as a variable, including JSON examples in your instructions. This:

In [ ]:
ChatPromptTemplate.from_template('Reply in JSON like {"a": 1} for {topic}')

   registers `a` as an input variable and fails at call time (LangChain's docs call this `INVALID_PROMPT_INPUT`). Escape literal braces by doubling them:

In [ ]:
ChatPromptTemplate.from_template('Reply in JSON like {{"a": 1}} for {topic}')
# input_variables == ['topic'], renders as: Reply in JSON like {"a": 1} for ...

2. **Missing variables fail loudly, extra inputs are ignored.** Call `.format_messages()` without a required key and you get a `KeyError` naming the missing variable. Pass keys the template does not use and nothing happens, they are silently dropped. So when a call fails, check the missing side; when a value seems to have no effect, check you spelled the key exactly as the template has it.
3. **The template format is f-string by default.** `template_format="jinja2"` is available for loops and conditionals, but it needs `pip install jinja2` (it is not a default dependency). Stick with f-string unless you genuinely need template logic.
4. **`format_messages` vs `invoke` vs `format`.** `.format_messages(**kwargs)` returns the message list (useful for inspection and for feeding non-runnable consumers). `.invoke(dict)` is the runnable call, same thing plus the runnable machinery. `.format(**kwargs)` returns the legacy single-string rendering, mostly useful for debugging. When a prompt misbehaves, print `prompt.format_messages(...)` before blaming the model.

## What can go wrong

- **JSON examples silently become variables.** The single most common prompt bug: an example JSON blob in the system message, unescaped braces, and a confusing missing-variable error at runtime. Double the braces in any literal `{`/`}` in your instructions.
- **Typos in partial keys fail silently.** `.partial()` does not check that its keys exist in the template, so `partial(assitant_name="Ada")` leaves `{assistant_name}` unfilled and you get a `KeyError` far from the real mistake. After partialling, print `input_variables` and confirm only the intended keys remain.
- **Role misuse across providers.** Putting instructions in an `ai` turn that one provider treats as a real prior answer and another ignores can change behaviour when you swap models. Keep the contract in `system`, the task in `human`, and use `ai` turns only for examples and confirmations.
- **Few-shot bloat.** Every example costs tokens on every call, and stale examples teach stale patterns. Review your examples when the task drifts; five sharp ones beat fifteen tired ones.
- **History placeholder with the wrong shape.** `MessagesPlaceholder("history")` expects a list of message objects (or dicts/tuples it can convert), not a string. Passing a pre-joined string of past turns produces one giant human message and the model loses the turn structure.

## Check your understanding

**1. What is the difference between `from_template` and `from_messages`, and when do you pick each?**
`from_template` builds a single human message with variables: the quick path for one-turn prompts. `from_messages` builds a full message list from roles, message objects, placeholders, and nested templates: the path for anything with a system contract, examples, history, or multiple turns.

**2. Why can the same template be reused across OpenAI, Anthropic, and Ollama without changes?**
The template renders to plain provider-neutral message objects, and the model is a separate runnable that consumes them. The template is the stable layer; only the model runnable is swapped. The one caveat is provider quirks around roles (e.g. how system messages are handled), so keep role usage conventional.

**3. What does `.partial()` return, and what is the callable trick?**
A new template with some variables pre-filled; the original is unchanged. Values may be callables, which are evaluated fresh on each format call, so `partial(today=lambda: date.today().isoformat())` always renders the current date.

**4. You add a JSON example to your system instructions and start getting a missing-variable error naming a key from inside the JSON. What happened and how do you fix it?**
The braces in the JSON example were read as template variables (`INVALID_PROMPT_INPUT`). Double every literal brace: `{{` and `}}`, so the example renders as plain JSON and only your real variables remain.

**5. How does `FewShotChatMessagePromptTemplate` compose with a larger template, and what does each example dict need?**
You build it from an `example_prompt` sub-template plus a list of example dicts, then place it as one entry inside `ChatPromptTemplate.from_messages(...)`. Each dict's keys fill the sub-template's variables, and the block renders as alternating human/ai messages before the real user turn.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.